In [45]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imports import *
from config import dir_config, main_config
from src.glm_hmm.cv_utils import bernoulli_null_loglik, pooled_bits_per_trial, select_best_n_states
from src.glm_hmm.fitting_utils import session_wise_fit, group_wise_fit

In [47]:
processed_dir = Path(dir_config.data.processed)
glm_hmm_dir = processed_dir / "glm_hmm"

MODEL_PATHS = {f.stem: f for f in glm_hmm_dir.iterdir() if f.is_dir()}

In [48]:
# --- model selection (mirrors 5.20): only used as a fallback if a bundle has no stored best_k ---
def session_bits(bundle):
    """Session-wise CV: bits/trial computed per session then averaged (mean, SEM across sessions)."""
    sw = bundle["session_wise"]
    states = np.array(list(sw["models"][0].keys()))
    n_trials, null = [], []
    for df in bundle["data"].values():
        valid = ~df["invalid_idx"].values
        y = df["choices"].values[valid]
        p = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        n_trials.append(int(valid.sum()))
        null.append((y * np.log(p) + (1 - y) * np.log(1 - p)).sum())
    n_trials, null = np.array(n_trials), np.array(null)
    bits = (sw["test_ll"].sum(2) - null[:, None]) / (n_trials[:, None] * np.log(2))
    return states, bits.mean(0), bits.std(0) / np.sqrt(bits.shape[0])


def pooled_bits(bundle):
    """Pooled CV: (states, mean_bits, sem_bits) over one held-out pass, vs the Bernoulli null."""
    gpc = bundle["group_pooled_cv"]
    null_ll, n_valid = bernoulli_null_loglik(bundle["data"])
    mean_bits, sem_bits = pooled_bits_per_trial(gpc["test_ll"], gpc["n_test_trials"], null_ll, n_valid)
    return np.asarray(gpc["state_range"]), mean_bits, sem_bits


def parsimonious_k(states, mean_bits, sem_bits):
    """Smallest K within one fold-SEM of the peak; mirrors cv_utils.select_best_n_states."""
    best_idx = int(np.argmax(mean_bits))
    cutoff = mean_bits[best_idx] - sem_bits[best_idx]
    chosen_idx = next(i for i in range(best_idx + 1) if mean_bits[i] >= cutoff)
    return int(states[chosen_idx]), int(states[best_idx])


def get_best_k(bundle, cv_type):
    """Use the best_k written by 5.20; recompute via the 1-SEM rule if it is absent."""
    if bundle.get("best_k") is not None:
        return int(bundle["best_k"])
    states, mean_bits, sem_bits = (pooled_bits if cv_type == "pooled_cv" else session_bits)(bundle)
    return parsimonious_k(states, mean_bits, sem_bits)[0]


# --- design matrices rebuilt from the saved per-session data frames ---
def session_arrays(bundle):
    """Per-session lists of (choices, inputs, mask) for the whole group, from the saved data."""
    feats = bundle["config"]["model_features"]
    observations, inputs, masks = [], [], []
    for df in bundle["data"].values():
        observations.append(df["choices"].values.reshape(-1, 1).astype(int))
        inputs.append(np.asarray(df[feats], dtype=float))
        masks.append(df["mask"].values.reshape(-1, 1).astype(bool))
    return observations, inputs, masks


# --- finetuning: one model PER SESSION at best_k, warm-started from the best CV fold ---
# Both CV types fit per session; they differ only in where the init comes from.
def _fit_per_session(bundle, best_k, init_params, best_folds, n_iters):
    """Refit one model per session at best_k from the given per-session init_params."""
    sessions = list(bundle["data"].keys())
    observations, inputs, masks = session_arrays(bundle)
    models_session, fit_ll_session = session_wise_fit(observations, inputs, masks, n_sessions=len(sessions), init_params=init_params, n_states=best_k, n_iters=n_iters)
    models = {session: models_session[idx] for idx, session in enumerate(sessions)}
    fit_lls = {session: fit_ll_session[idx] for idx, session in enumerate(sessions)}
    return {"models": models, "fit_lls": fit_lls, "best_folds": best_folds}


def finetune_session(bundle, best_k, n_iters=2500):
    """Per-session models; each session warm-started from its own best-generalizing CV fold."""
    sw = bundle["session_wise"]
    sessions = list(bundle["data"].keys())
    state_idx = list(sw["models"][0].keys()).index(best_k)

    init_params = {"glm_weights": {}, "transition_matrices": {}}
    best_folds = {}
    for idx, session in enumerate(sessions):
        best_fold = int(np.nanargmax(sw["test_ll"][idx, state_idx, :]))
        m = sw["models"][idx][best_k][best_fold]
        init_params["glm_weights"][idx] = m.observations.params
        init_params["transition_matrices"][idx] = m.transitions.params
        best_folds[session] = best_fold
    return _fit_per_session(bundle, best_k, init_params, best_folds, n_iters)


def finetune_pooled(bundle, best_k, n_iters=2500):
    """Per-session models AND one pooled model, all warm-started from the group's best pooled fold.

    Per-session fits give per-session deliverables (noisy, data-starved); the extra pooled fit over
    all the group's sessions gives clean group-level weights, stored under "pooled".
    """
    gpc = bundle["group_pooled_cv"]
    state_idx = int(np.where(np.asarray(gpc["state_range"]) == best_k)[0][0])
    best_fold = int(np.nanargmax(gpc["test_ll"][state_idx]))  # one best fold for the whole group
    init = gpc["models"][best_k][best_fold]
    init_flat = {"glm_weights": init.observations.params, "transition_matrices": init.transitions.params}

    sessions = list(bundle["data"].keys())
    init_params = {
        "glm_weights": {idx: init.observations.params for idx in range(len(sessions))},
        "transition_matrices": {idx: init.transitions.params for idx in range(len(sessions))},
    }
    best_folds = {session: best_fold for session in sessions}
    result = _fit_per_session(bundle, best_k, init_params, best_folds, n_iters)

    # Also fit one pooled model over all the group's sessions (clean group-level weights).
    observations, inputs, masks = session_arrays(bundle)
    pooled_model, pooled_fit_ll = group_wise_fit(observations, inputs, masks, init_flat, n_states=best_k, n_iters=n_iters)
    result["pooled"] = pooled_model
    result["pooled_fit_ll"] = pooled_fit_ll
    return result

## Finetune the selected model on the full data

For each model/group, take the `best_k` chosen in `5.20` and refit **one model per session** on the
full data (`session_wise_fit`), warm-started from the best CV fold. The two CV types differ in the
init source:

- **session-CV**: each session is initialized from its **own** best-generalizing fold.
- **pooled-CV**: the group's single best pooled fold (best fold at `best_k`) initializes **every**
  session; additionally a **single pooled model** over all the group's sessions is fit
  (`group_wise_fit`) and stored under `"pooled"` — clean group-level weights, since per-session
  fits are data-starved for multi-state models.

Each group's models are saved with `data` and `config` beside the CV pickle as
`<model_dir>/<group>_final.pkl` (the loop skips these `_final` files on re-runs).

In [49]:
desired_order = ["asmHC", "Tremor_OFF", "Brady_OFF", "Tremor_ON", "Brady_ON"]
order_map = {name: i for i, name in enumerate(desired_order)}

N_ITER_FINETUNE = 2500

for model_name, model_path in MODEL_PATHS.items():
    cv_type = "pooled_cv" if model_name.endswith("__global_pooled_cv") else "session_cv"
    print(f"=== {model_name} [{cv_type}] ===")

    for subtype_model in sorted(model_path.glob("*.pkl"), key=lambda p: order_map.get(p.stem, float("inf"))):
        if subtype_model.stem.endswith("_final"):
            continue  # skip finetuned outputs from a previous run
        group_name = subtype_model.stem
        with open(subtype_model, "rb") as f:
            glm_hmm = pickle.load(f)

        best_k = get_best_k(glm_hmm, cv_type)
        finetune = finetune_pooled if cv_type == "pooled_cv" else finetune_session
        finetuned = finetune(glm_hmm, best_k, n_iters=N_ITER_FINETUNE)  # {models, fit_lls, best_folds[, pooled]}

        # Store the finetuned models beside the CV pickle, as <group>_final.pkl.
        models_and_data = {
            "model": finetuned,
            "best_k": best_k,
            "cv_type": cv_type,
            "data": glm_hmm["data"],
            "config": glm_hmm["config"],
        }
        out_path = subtype_model.parent / f"{subtype_model.stem}_final.pkl"
        with open(out_path, "wb") as f:
            pickle.dump(models_and_data, f)

        extra = " + pooled" if "pooled" in finetuned else ""
        print(f"  {group_name:12s} best_k={best_k}  ({len(finetuned['models'])} session models{extra}) -> {out_path.name}")

=== masked_with_bias_1_back_prev_choice_coherence_standardized_stimulus__global_pooled_cv [pooled_cv] ===


Converged to LP: -423.5:  16%|█▌        | 390/2500 [00:01<00:08, 237.03it/s]


  0%|          | 0/2500 [00:00<?, ?it/s]

  asmHC        best_k=3  (18 session models + pooled) -> asmHC_final.pkl


Converged to LP: -288.9:   8%|▊         | 193/2500 [00:00<00:06, 345.40it/s]


...

  Brady_OFF    best_k=1  (10 session models) -> Brady_OFF_final.pkl


Converged to LP: -326.2:   0%|          | 2/2500 [00:00<00:25, 97.92it/s]


  Tremor_ON    best_k=1  (11 session models) -> Tremor_ON_final.pkl


Converged to LP: -335.2:   0%|          | 2/2500 [00:00<00:22, 112.53it/s]


  Brady_ON     best_k=1  (10 session models) -> Brady_ON_final.pkl


In [50]:
# Sanity check: reload and summarise the finetuned models saved beside each CV pickle.
for model_name, model_path in MODEL_PATHS.items():
    print(f"=== {model_name} ===")
    for path in sorted(model_path.glob("*_final.pkl"), key=lambda p: order_map.get(p.stem[: -len("_final")], float("inf"))):
        md = pickle.load(open(path, "rb"))
        models = md["model"]["models"]
        w = next(iter(models.values())).observations.params
        print(f"  {path.stem:18s} best_k={md['best_k']}  cv={md['cv_type']:10s} {len(models)} session models  weights{tuple(w.shape)}")

=== masked_with_bias_1_back_prev_choice_coherence_standardized_stimulus__global_pooled_cv ===
  asmHC_final        best_k=3  cv=pooled_cv  18 session models  weights(3, 1, 5)
  Tremor_OFF_final   best_k=2  cv=pooled_cv  11 session models  weights(2, 1, 5)
  Brady_OFF_final    best_k=2  cv=pooled_cv  10 session models  weights(2, 1, 5)
  Tremor_ON_final    best_k=3  cv=pooled_cv  11 session models  weights(3, 1, 5)
  Brady_ON_final     best_k=2  cv=pooled_cv  10 session models  weights(2, 1, 5)
=== masked_with_bias_1_back_prev_choice_coherence_standardized_stimulus__session_pooled_cv ===
  asmHC_final        best_k=1  cv=session_cv 18 session models  weights(1, 1, 5)
  Tremor_OFF_final   best_k=1  cv=session_cv 11 session models  weights(1, 1, 5)
  Brady_OFF_final    best_k=1  cv=session_cv 10 session models  weights(1, 1, 5)
  Tremor_ON_final    best_k=1  cv=session_cv 11 session models  weights(1, 1, 5)
  Brady_ON_final     best_k=1  cv=session_cv 10 session models  weights(1, 1, 5)
=